# STG-NF baseline (B3), trained and scored under CALM-VAD's own protocol

Closes a gap the paper's Baselines paragraph (`paper/main.tex`, search `B3`)
left open on purpose: B3 -- "a published pose-density score such as STG-NF's
own trained normalizing-flow model" -- was cited from the published paper
instead of reproduced, because "a fair run would mean cloning and training
that method's full codebase... a different scope of work." This notebook
*is* that scope of work: it clones the real STG-NF repo, trains it on
ShanghaiTech and UBnormal (its own two natively-supported benchmarks, and
the two this project already has verified test-split ground truth for),
and scores its output through `calm.external_scores` -- the exact same
event-F1 / FAPH / ECE protocol `calm.harness.run()` uses for CALM-VAD and
B0/B1/B2/B4 -- so the resulting numbers are a real head-to-head, not a
frame-AUC citation next to numbers computed a different way.

**STG-NF, verified:** official repo of Hirschorn & Avidan, *"Normalizing
Flows for Human Pose Anomaly Detection"*, ICCV 2023 --
<https://github.com/orhir/STG-NF> (author page: <https://orhir.github.io/>,
matches the paper's first author; arXiv 2211.10946; real checkpoints
committed: `ShanghaiTech_85_9.tar`, `UBnormal_unsupervised_71_8.tar`,
`UBnormal_supervised_79_2.tar` -- the numbers in those filenames are their
own published frame-AUC results, used below as a sanity check, not a
substitute for actually running it).

**Before you start:**
1. `Runtime -> Change runtime type -> T4 GPU` (required -- STG-NF's own
   code hardcodes `--device cuda:0` by default and this notebook stops
   early with a clear message if no GPU is allocated).
2. **Mount your Google Drive when the cell below asks** -- trained
   checkpoints and the extracted per-frame scores are saved there as each
   dataset finishes, so a Colab disconnect after ShanghaiTech is done
   doesn't cost you that training run again.
3. **Have `data/pose/shanghaitech.json` and `data/pose/ubnormal.json`
   ready on your own machine** (they're already there if you have this
   project checked out locally) -- Cell 4b will prompt you to upload them
   once (~300 MB total, cached to Drive after that so it's a one-time
   thing). These are `.gitignore`d, so `git clone` in Cell 1 does not
   bring them over, and Cell 9's comparison needs this project's own
   already-verified copy specifically, not a fresh regeneration.
4. Run cells top to bottom, once. Re-running is safe: a dataset whose
   checkpoint is already on Drive is skipped, not retrained.

**Honesty notes baked into this notebook (read before trusting a number):**
- Training runtime is **estimated, not measured** -- I have no GPU in the
  environment that built this notebook, so I could not time it myself. My
  reasoning (stated where relevant): STG-NF trains on pose sequences only
  (17 keypoints x a short window), no video decoding, no CNN backbone, and
  the model itself is very small (its own `calc_num_of_params` print will
  show you the real number when you run it) -- so my best guess is well
  under an hour per dataset on a T4, and this notebook prints wall-clock
  time so you can tell me if that guess was wrong.
- Two concrete, verified (not guessed) incompatibilities between STG-NF's
  2022-era pinned environment and a modern Colab runtime are patched below,
  with the exact file/line and the exact library-version reason for each.
  If you hit a *different* incompatibility, that's real new information,
  not something this notebook silently papered over.
- STG-NF's own training loop does not support resuming mid-training from a
  checkpoint (verified by reading `models/training.py`: `Trainer.train()`
  always starts its epoch loop at 0). Resumability here is coarser than
  the other two Colab notebooks in this repo: a *finished* dataset's
  checkpoint is never retrained, but a disconnect *during* one dataset's
  training loses that dataset's progress (default 8 epochs -- see the
  runtime note above for why that's expected to be short).


## Cell 1 -- install + clone this project (for `calm.*` scoring code)

In [ ]:
import os
# Guard against re-running this cell after a previous run already chdir()'d into
# /content/sentrix: if this kernel's cwd no longer exists on disk, every shell
# subprocess below breaks ('getcwd() failed' / 'shell-init: error retrieving
# current directory'). Move somewhere that's never deleted, first.
os.chdir('/content')

!pip -q install numpy scipy scikit-learn pyyaml gdown

import os, sys, subprocess, time

def sh(cmd, check=False, cwd=None):
    '''Run a shell command with output visible (never silently swallowed),
    so a failure here prints the real reason instead of a confusing error
    two lines later. For short commands (installs, clones, listings).'''
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.stdout.strip():
        print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')
    return r

def sh_stream(cmd, cwd=None):
    '''Like sh(), but for LONG-running commands (STG-NF's own train_eval.py
    prints tqdm progress bars as it trains). subprocess.run(capture_output=
    True) buffers everything until the process exits -- on a multi-minute
    training loop that means zero visible progress until the very end, which
    is exactly the 'silent failure' shape this project's Colab notebooks are
    built to avoid. This streams output line by line instead, and still
    raises with the real error (last 40 lines) on a non-zero exit.'''
    print(f'$ {cmd}' + (f'   (cwd={cwd})' if cwd else ''))
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'command failed (exit {proc.returncode}): {cmd}\n'
                            '--- last 40 lines of output ---\n' + ''.join(lines[-40:]))
    return ''.join(lines)

REPO_URL = 'https://github.com/FaizanAbbas512/Sentrix.git'
for attempt in range(1, 4):
    os.chdir('/content')   # never rm -rf the directory this process is standing in
    sh('rm -rf /content/sentrix')
    r = sh('git clone --depth 1 %s /content/sentrix' % REPO_URL)
    if os.path.isdir('/content/sentrix/calm'):
        break
    print(f'clone attempt {attempt} did not produce /content/sentrix/calm -- retrying...')
else:
    raise RuntimeError('git clone failed 3 times -- check the error output above '
                        '(often a transient Colab network hiccup: Runtime -> '
                        'Restart session, then re-run this cell).')

os.chdir('/content/sentrix'); sys.path.insert(0, '/content/sentrix')
sh('python -m calm.selftest | tail -3')

import torch
print('torch version:', torch.__version__, '| cuda available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    hw = sh('nvidia-smi --query-gpu=name --format=csv,noheader')
    gpu_hw_present = hw.returncode == 0 and hw.stdout.strip()
    if gpu_hw_present:
        print(f'GPU hardware IS present ({hw.stdout.strip()}) but this torch build has no')
        print('CUDA support. Reinstalling the correct CUDA build...')
        sh('pip -q install --force-reinstall torch --index-url https://download.pytorch.org/whl/cu124')
        print('Reinstalled -- torch cannot be reliably reloaded into an already-running')
        print('process. Runtime -> Restart session, then re-run this cell (Cell 1) again.')
    else:
        print('No GPU hardware allocated to this runtime at all.')
        print('Fix: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.')

GPU_OK = torch.cuda.is_available()
print('\nGPU ready for STG-NF training:', GPU_OK)
if not GPU_OK:
    print('!! STG-NF hardcodes --device cuda:0 by default in its own args.py -- training')
    print('   cells below will fail fast (not hang) until this is fixed.')


## Cell 2 -- get STG-NF's real, official code

Cloning the paper authors' own repo (verified above: matches the paper, has
real commit/release history, ships their own trained checkpoints). Two
patches are applied to the clone -- not to change any training/scoring
logic, only to fix two concrete breakages between STG-NF's 2022-pinned
`environment.yml` (`numpy=1.23.3`, `matplotlib=3.5.3`, `python=3.8`,
`torch=1.10.1`) and the current Colab runtime, found by reading the source
(not guessed):

1. **`dataset.py:250`** and **`utils/pose_utils.py:166`** both do
   `.astype(np.int)` / `dtype=np.int`. `np.int` was a deprecated alias for
   the builtin `int` and was **removed** in NumPy 1.24 (released Dec 2023).
   Any numpy Colab ships in 2026 is well past that -- this is a hard
   `AttributeError` at import time, not a maybe.
2. **`utils/pose_utils.py:9`** does `plt.style.use('seaborn-ticks')` at
   **module import time**. That style name was renamed to
   `seaborn-v0_8-ticks` in Matplotlib 3.6 (Sept 2022) and the old name was
   removed -- also a hard error at import, before any training code runs,
   even though this project's actual pipeline never calls any of
   `pose_utils.py`'s plotting functions (only used for `--plot_vid`,
   default off).

We deliberately do **not** try to recreate the pinned conda environment
(slow, and STG-NF's actual model/training code is plain PyTorch -- nothing
in it needs CUDA 10.2 specifically). If training hits a *different*
incompatibility once you actually run this, that's new information worth
reporting back, not something this notebook silently worked around.

In [ ]:
STGNF_DIR = '/content/STG-NF'
STGNF_URL = 'https://github.com/orhir/STG-NF.git'

for attempt in range(1, 4):
    sh(f'rm -rf {STGNF_DIR}')
    sh(f'git clone --depth 1 {STGNF_URL} {STGNF_DIR}')
    if os.path.isfile(f'{STGNF_DIR}/train_eval.py'):
        break
    print(f'clone attempt {attempt} did not produce train_eval.py -- retrying...')
else:
    raise RuntimeError('git clone of STG-NF failed 3 times -- check the error output above.')

for f in ('models/STG_NF/model_pose.py', 'utils/scoring_utils.py', 'args.py', 'dataset.py'):
    assert os.path.isfile(f'{STGNF_DIR}/{f}'), f'expected file missing after clone: {f}'
print('STG-NF cloned OK, key files present.')

# ---- patch 1: np.int (removed in numpy>=1.24) -----------------------------
import re
for relpath, pattern, replacement in [
    ('dataset.py', r'dtype=np\.int\)', 'dtype=int)'),
    ('utils/pose_utils.py', r'astype\(np\.int\)', 'astype(int)'),
]:
    p = f'{STGNF_DIR}/{relpath}'
    with open(p, 'r', encoding='utf-8') as fh:
        src = fh.read()
    new_src, n = re.subn(pattern, replacement, src)
    assert n >= 1, (f'expected to find {pattern!r} in {relpath} but did not -- STG-NF may '
                     f'have changed since this notebook was written; check {relpath} by hand')
    with open(p, 'w', encoding='utf-8') as fh:
        fh.write(new_src)
    print(f'patched {relpath}: {n} occurrence(s) of np.int -> int')

# ---- patch 2: seaborn-ticks style removed in matplotlib>=3.6 --------------
p = f'{STGNF_DIR}/utils/pose_utils.py'
with open(p, 'r', encoding='utf-8') as fh:
    src = fh.read()
old = "plt.style.use('seaborn-ticks')"
assert old in src, ('expected the exact seaborn-ticks style.use() line in '
                     'utils/pose_utils.py but did not find it -- check by hand')
new_src = src.replace(
    old,
    "try:\n"
    "    plt.style.use('seaborn-ticks')\n"
    "except OSError:\n"
    "    try:\n"
    "        plt.style.use('seaborn-v0_8-ticks')   # renamed in matplotlib>=3.6\n"
    "    except OSError:\n"
    "        pass   # cosmetic only -- none of this project's calls touch plotting",
)
with open(p, 'w', encoding='utf-8') as fh:
    fh.write(new_src)
print('patched utils/pose_utils.py: seaborn-ticks style.use() now falls back safely')

# ---- remaining pip deps STG-NF needs that Cell 1 didn't already install ---
sh('pip -q install matplotlib tensorboard tqdm pillow')
print('\nSTG-NF ready at', STGNF_DIR)


## Cell 3 -- mount Drive (trained checkpoints + extracted scores persist here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSIST = '/content/drive/MyDrive/stgnf_baseline'
CKPT_DIR = f'{PERSIST}/checkpoints'
SCORES_DIR = f'{PERSIST}/scores'
for d in (CKPT_DIR, SCORES_DIR):
    os.makedirs(d, exist_ok=True)
print('checkpoints ->', CKPT_DIR)
print('scores      ->', SCORES_DIR)


## Cell 4 -- get STG-NF's own pose + ground-truth data

This is the **same Google Drive file** this project's own
`colab/calm_vad_colab.ipynb` already downloads for the ShanghaiTech and
UBnormal entries in its `DATASETS` registry (`gdrive_file =
'1o9h3Kh6zovW4FIHpNBGnYIRSbGCu-qPt'`) -- it is STG-NF's own official
"Data Directory" download, linked from their README, and it is where this
project's `data/pose/shanghaitech.json` and `data/pose/ubnormal.json` were
themselves converted *from* in the first place (see
`calm/datasets.py`'s `load_gepc_json` / `load_ubnormal_stgnf`, whose
docstrings say "STG-NF release" explicitly).

**Why pull this bundle instead of converting our own already-consolidated
`data/pose/{shanghaitech,ubnormal}.json` back into STG-NF's input format:**
1. Our JSON only ever stored the **test** split (that's all CALM-VAD's own
   evaluation needs) -- training STG-NF needs the **train** split too
   (ShanghaiTech's normal-only training cameras; UBnormal's 186
   normal-by-construction training clips), which we never extracted.
2. STG-NF's own per-frame record needs a `"scores"` field (a per-frame
   detection-confidence scalar) that our consolidated JSON does not keep
   (only per-keypoint confidence survives the conversion) -- reconstructing
   it would be a guess. Pulling the original bundle sidesteps that
   entirely: it's the literal file their own pipeline was built around.

The ground truth used for the **actual comparison numbers** later in this
notebook still comes from this project's own, already-verified
`data/pose/{shanghaitech,ubnormal}.json` -- not from this bundle's copy --
specifically to avoid re-importing either of two bugs this project already
found and fixed once in its own history (a ShanghaiTech duplicate-clip bug,
and a UBnormal ground-truth-polarity inversion). STG-NF's bundle supplies
the **pose input for training** and, via its own scoring code, the
**per-clip anomaly score**; our own file supplies every label used to
score it. The next cell (Cell 5) cross-checks that the two sources agree on
which clips exist before anything is trusted.

In [ ]:
BUNDLE_ZIP = '/content/stgnf_data_bundle.zip'
BUNDLE_DIR = '/content/stgnf_data_bundle'
GDRIVE_FILE_ID = '1o9h3Kh6zovW4FIHpNBGnYIRSbGCu-qPt'   # same id calm_vad_colab.ipynb uses

if not os.path.isfile(BUNDLE_ZIP):
    import gdown
    for attempt in range(1, 4):
        try:
            gdown.download(f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}',
                            BUNDLE_ZIP, quiet=False, fuzzy=True)
            if os.path.isfile(BUNDLE_ZIP) and os.path.getsize(BUNDLE_ZIP) > 0:
                break
        except Exception as e:
            print(f'download attempt {attempt} failed: {e}')
        time.sleep(5)
    else:
        raise RuntimeError('gdown failed 3 times -- if Google is rate-limiting anonymous '
                            'downloads from this Colab IP, open '
                            f'https://drive.google.com/file/d/{GDRIVE_FILE_ID}/view in your '
                            'browser, save it to your own Drive, and point BUNDLE_ZIP at that '
                            'path instead.')
else:
    print('bundle zip already present, skipping download')

if not os.path.isdir(BUNDLE_DIR) or not os.listdir(BUNDLE_DIR):
    sh(f'mkdir -p {BUNDLE_DIR}', check=True)
    r = sh(f'unzip -q -o "{BUNDLE_ZIP}" -d {BUNDLE_DIR}')
    if r.returncode != 0:
        # not every archive gdown pulls down is a plain zip -- fall back to 7z, which
        # reads zip/rar/7z/tar transparently
        sh('apt-get -qq install -y p7zip-full > /dev/null')
        sh(f'7z x "{BUNDLE_ZIP}" -o{BUNDLE_DIR} -y', check=True)
print('bundle extracted to', BUNDLE_DIR)

# ---- locate the ShanghaiTech/ and UBnormal/ folders inside the extracted tree
# (bounded-depth search, not a blind hardcoded path -- the zip's exact top-level
# wrapping folder is not guaranteed, but a directory literally named
# 'ShanghaiTech' / 'UBnormal' containing a 'pose' and 'gt' subfolder is what
# STG-NF's own README documents, so that's what we look for)
def find_dataset_dir(root, name, timeout_s=180):
    start = time.time()
    for dirpath, dirnames, filenames in os.walk(root):
        if time.time() - start > timeout_s:
            return None
        if os.path.basename(dirpath) == name and 'pose' in dirnames and 'gt' in dirnames:
            return dirpath
    return None

st_src = find_dataset_dir(BUNDLE_DIR, 'ShanghaiTech')
ub_src = find_dataset_dir(BUNDLE_DIR, 'UBnormal')
assert st_src, f'could not find a ShanghaiTech/ folder with pose/+gt/ under {BUNDLE_DIR}'
assert ub_src, f'could not find a UBnormal/ folder with pose/+gt/ under {BUNDLE_DIR}'
print('ShanghaiTech source:', st_src)
print('UBnormal source    :', ub_src)

# ---- lay out exactly where STG-NF's own code expects it: <STGNF_DIR>/data/<Dataset>
# (get_dataset_scores in utils/scoring_utils.py hardcodes relative paths like
# 'data/ShanghaiTech/gt/test_frame_mask/' -- this is not just convenient, it's required)
#
# Self-healing, not just 'skip if anything is already there': a STALE or BROKEN
# symlink from an earlier run (e.g. one whose target no longer exists after a
# Colab restart wiped /content/ but somehow left this dangling link behind) used
# to be silently kept (os.path.lexists() is True for a broken symlink too), which
# is exactly what produced a real 'directory does not exist' failure two cells
# later instead of here where it's actually fixable. Now: only reuse an existing
# data/<Dataset> path if it resolves to a real, non-empty directory; otherwise
# remove whatever is there (broken link, empty dir, stray file) and relink fresh.
import shutil as _shutil
os.makedirs(f'{STGNF_DIR}/data', exist_ok=True)
for name, src in (('ShanghaiTech', st_src), ('UBnormal', ub_src)):
    dst = f'{STGNF_DIR}/data/{name}'
    if os.path.isdir(dst) and os.listdir(dst):
        print(f'{dst} already links to a real, non-empty directory -- reusing as-is')
        continue
    if os.path.islink(dst) or os.path.exists(dst):
        print(f'{dst} exists but is broken or empty (stale from an earlier run) -- removing it')
        if os.path.islink(dst) or os.path.isfile(dst):
            os.remove(dst)
        else:
            _shutil.rmtree(dst)
    os.symlink(src, dst, target_is_directory=True)
    print(f'linked {dst} -> {src}  ({len(os.listdir(dst))} entries: {os.listdir(dst)[:5]})')

print()
print('pose/gt data ready at', f'{STGNF_DIR}/data/ShanghaiTech', 'and',
      f'{STGNF_DIR}/data/UBnormal')


## Cell 4b -- get THIS PROJECT's own already-verified pose/ground-truth JSON

Cell 4 downloaded STG-NF's **own** raw pose/GT bundle (what STG-NF trains
on). This is a separate, smaller pair of files: `data/pose/shanghaitech.json`
and `data/pose/ubnormal.json` -- the exact same, already-verified files every
other number in this paper (CALM-VAD, B0-B4) is scored against, including a
ShanghaiTech duplicate-clip bug this project found and fixed once already in
its own history. Cell 9 needs these specific files, not a fresh conversion of
STG-NF's bundle, so the STG-NF comparison lines up against the identical
ground truth as everything else -- regenerating them from the STG-NF bundle
here risks silently reintroducing a bug already fixed, or a subtly different
result than what's already published.

These two files total **~300 MB** (103 MB + 195 MB) and are `.gitignore`d in
this project (too big for git), so `git clone` in Cell 1 did not bring them
over -- they have to come from you, once. This cell:
1. Checks if they're already in this Colab session (skips if so).
2. Checks your Drive at `{PERSIST}/pose_data/` (skips the upload if a
   previous run already cached them there).
3. Otherwise, prompts you to upload each file directly (a one-time
   ~300 MB browser upload) and caches it to that Drive path so you never
   need to upload it again on a future Colab session.

**~300 MB is about 2% of Google's free 15 GB Drive tier** -- small next to
the multi-hundred-GB video archives this project's *other* Colab notebooks
had to deal with (see `colab/calm_vad_nwpu_colab.ipynb`'s history); nothing
bigger than this ever touches your Drive in this notebook.

In [ ]:
import shutil
from google.colab import files as _colab_files

os.makedirs('data/pose', exist_ok=True)   # empty dir, never tracked by git -- clone doesn't create it

POSE_DRIVE_DIR = f'{PERSIST}/pose_data'
os.makedirs(POSE_DRIVE_DIR, exist_ok=True)

REQUIRED_POSE_FILES = {
    'data/pose/shanghaitech.json': f'{POSE_DRIVE_DIR}/shanghaitech.json',
    'data/pose/ubnormal.json': f'{POSE_DRIVE_DIR}/ubnormal.json',
}
EXPECTED_MB = {
    'data/pose/shanghaitech.json': 103,
    'data/pose/ubnormal.json': 195,
}

for repo_path, drive_path in REQUIRED_POSE_FILES.items():
    if os.path.isfile(repo_path) and os.path.getsize(repo_path) > 0:
        mb = os.path.getsize(repo_path) / 1e6
        print(f'{repo_path}: already present in this session ({mb:.0f} MB)')
        continue

    if os.path.isfile(drive_path) and os.path.getsize(drive_path) > 0:
        shutil.copy(drive_path, repo_path)
        mb = os.path.getsize(repo_path) / 1e6
        print(f'{repo_path}: copied from Drive cache ({mb:.0f} MB) -- no upload needed')
        continue

    # a previous attempt in this same session may already have uploaded the
    # file into the working directory (e.g. if a later step failed after the
    # upload but before this file was filed away) -- reuse it, don't re-ask
    loose_name = os.path.basename(repo_path)
    if os.path.isfile(loose_name) and os.path.getsize(loose_name) > 0:
        print(f'{loose_name}: found already-uploaded copy in the working directory -- reusing it')
    else:
        print()
        print(f'{repo_path} not found in this session or on Drive.')
        print('This is YOUR local file (from your own machine, where the rest of this')
        print('project already lives) -- upload it now using the file picker that appears.')
        print(f'(expected: roughly {EXPECTED_MB[repo_path]} MB)')
        uploaded = _colab_files.upload()
        assert uploaded, f'no file uploaded -- re-run this cell and pick {loose_name}'
        loose_name = next(iter(uploaded.keys()))

    shutil.move(loose_name, repo_path)
    shutil.copy(repo_path, drive_path)
    mb = os.path.getsize(repo_path) / 1e6
    print(f'saved -> {repo_path} ({mb:.0f} MB), cached -> {drive_path}')
    print('(future Colab sessions will reuse this, no re-upload)')

for repo_path in REQUIRED_POSE_FILES:
    ok = os.path.isfile(repo_path) and os.path.getsize(repo_path) > 0
    assert ok, f'{repo_path} still missing/empty after fetch attempt'

print()
print('All pose/ground-truth files ready.')


## Cell 5 -- sanity check: does STG-NF's own test-clip list match ours?

CPU-only, no GPU/torch needed, runs before any training. This project has
twice already found a real bug from exactly this kind of mismatch (a
duplicate-clip bug in ShanghaiTech's pose conversion, and an inverted
ground-truth polarity in UBnormal's) -- so before trusting anything trained
against this bundle, we check that the bundle's own test-clip name set and
`data/pose/{shanghaitech,ubnormal}.json`'s clip names line up 1:1 by name.

If this cell's assertions fail, **stop** -- something about the download or
the clip-naming assumptions in this notebook is wrong, and training on top
of a silent misalignment would produce numbers that look plausible but
compare the wrong clips.

In [ ]:
from calm import datasets as D

def _tree(root, max_lines=40):
    if not os.path.isdir(root):
        return ['(this directory does not exist: %s)' % root]
    out = []
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        out.append(('  ' * depth) + os.path.basename(dirpath) + '/')
        for fn in filenames[:3]:
            out.append(('  ' * (depth + 1)) + fn)
        if len(filenames) > 3:
            out.append(('  ' * (depth + 1)) + ('... (%d files total)' % len(filenames)))
        if len(out) > max_lines:
            out.append('... (truncated)')
            break
    return out

def _find_files_dir(root, suffix):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        if any(fn.endswith(suffix) for fn in filenames):
            hits.append(dirpath)
    return hits

def stgnf_test_clip_names(dataset):
    if dataset == 'UBnormal':
        pose_root = f'{STGNF_DIR}/data/UBnormal/pose'
        hits = _find_files_dir(pose_root, '_alphapose_tracked_person.json')
        if not hits:
            print('no *_alphapose_tracked_person.json files found anywhere under', pose_root)
            print('actual tree:')
            for line in _tree(pose_root):
                print(line)
            raise AssertionError('see printed tree above')
        test_hits = [d for d in hits if 'test' in d.lower() and 'train' not in d.lower()]
        pool = test_hits or hits
        pose_test = max(pool, key=lambda d: len(
            [f for f in os.listdir(d) if f.endswith('_alphapose_tracked_person.json')]))
        print('  (UBnormal test-pose dir resolved to:', pose_test, ')')
        names = sorted(fn.replace('_alphapose_tracked_person.json', '')
                        for fn in os.listdir(pose_test) if fn.endswith('.json'))
    else:
        gt_root = f'{STGNF_DIR}/data/ShanghaiTech/gt'
        hits = _find_files_dir(gt_root, '.npy')
        if not hits:
            print('no .npy frame-mask files found anywhere under', gt_root)
            print('actual tree:')
            for line in _tree(gt_root):
                print(line)
            raise AssertionError('see printed tree above')
        test_hits = [d for d in hits if 'test' in d.lower()]
        pool = test_hits or hits
        gt_dir = max(pool, key=lambda d: len([f for f in os.listdir(d) if f.endswith('.npy')]))
        print('  (ShanghaiTech test-gt dir resolved to:', gt_dir, ')')
        names = sorted(fn.replace('.npy', '') for fn in os.listdir(gt_dir) if fn.endswith('.npy'))
    return set(names)

for key, dataset, clips_json in (('shanghaitech', 'ShanghaiTech', 'data/pose/shanghaitech.json'),
                                  ('ubnormal', 'UBnormal', 'data/pose/ubnormal.json')):
    bundle_names = stgnf_test_clip_names(dataset)
    ours = {c.name for c in D.load_generic_json(clips_json)}
    only_bundle = bundle_names - ours
    only_ours = ours - bundle_names
    print(f'--- {key} ---')
    print(f'  bundle test clips : {len(bundle_names)}')
    print(f'  our test clips    : {len(ours)}')
    print(f'  in bundle only    : {len(only_bundle)}  {sorted(only_bundle)[:5]}')
    print(f'  in ours only      : {len(only_ours)}  {sorted(only_ours)[:5]}')
    overlap = len(bundle_names & ours) / max(len(bundle_names | ours), 1)
    print(f'  name overlap      : {overlap:.1%}')
    assert overlap > 0.90, key + ': bundle and our clip-name sets diverge by more than 10%% -- see diffs above'
print()
print('OK -- STG-NF bundle and this project own ground truth agree on clip identity.')


## Cell 6 -- what gets trained

`STGNF_DATASETS` records, per dataset, the one flag STG-NF's own README
departs from its default for (`--seg_len`), and the published frame-AUC
baked into that dataset's official checkpoint filename (used later purely
as a sanity check, not a substitute for the real run).

**`seg_len=16` for UBnormal, reasoned not guessed:** `seg_len` fixes part of
STG-NF's model architecture (a temporal squeeze factor), so a checkpoint can
only be *evaluated* with the same `seg_len` it was *trained* with -- and
STG-NF's README evaluates both published UBnormal checkpoints with
`--seg_len 16` (`checkpoints/UBnormal_unsupervised_71_8.tar` and
`..._supervised_79_2.tar`). That is strong evidence their own UBnormal
training used `seg_len=16` too, since a mismatched value would make loading
those checkpoints fail outright. ShanghaiTech's README commands never
override `seg_len`, so we don't either (STG-NF's own default, 24).

**Unsupervised only, on purpose:** STG-NF supports an optional "supervised"
mode (`--pose_path_train_abnormal` + `--R`) that also trains on labeled
abnormal clips. We use the plain **unsupervised** variant for both datasets
-- the fair match to CALM-VAD and B0/B1/B2/B4, none of which see anomaly
labels during training either (CALM-VAD's calibration split uses labels,
but that is a post-hoc *evaluation-time* step, not training the detector
itself).

In [ ]:
STGNF_DATASETS = {
    'shanghaitech': dict(
        stgnf_name='ShanghaiTech',
        seg_len=None,          # STG-NF's own default (24); README never overrides it here
        clips_json='data/pose/shanghaitech.json',
        published_auc=85.9,    # checkpoints/ShanghaiTech_85_9.tar (unsupervised -- ShanghaiTech has no other mode)
        tag='stgnf_shanghaitech',
        label='STG-NF (B3, reproduced)',
    ),
    'ubnormal': dict(
        stgnf_name='UBnormal',
        seg_len=16,              # see markdown above for why this is reasoned, not guessed
        clips_json='data/pose/ubnormal.json',
        published_auc=71.8,      # checkpoints/UBnormal_unsupervised_71_8.tar -- unsupervised variant
        tag='stgnf_ubnormal',
        label='STG-NF (B3, reproduced)',
    ),
}
for k, v in STGNF_DATASETS.items():
    print(k, '->', v['stgnf_name'], f"(seg_len={v['seg_len'] or 'default'})")


## Cell 7 -- the score-dump script

STG-NF's own `train_eval.py` trains, evaluates, and prints a single AUC
number -- it never saves the per-clip, per-frame score array
`utils/scoring_utils.get_dataset_scores()` computes internally to get there.
`sentrix_score_dump.py`, written below (not part of STG-NF's own repo), is
that same sequence with about 15 added lines: load a checkpoint instead of
training (fast, GPU inference only), call STG-NF's own `get_dataset_scores`
+ `smooth_scores` unmodified (the exact call `train_eval.py` makes
internally to compute its printed AUC), recover each score array's clip
name from the matching ground-truth filename list (same convention this
project's own `calm/datasets.py` already documents for this exact release),
sign-flip (STG-NF's raw output is a *normality* log-likelihood -- high
means normal -- so we negate it to match `calm`'s "higher = more anomalous"
convention used everywhere else in this project), and write one JSON per
dataset.

Training itself is done with STG-NF's own `train_eval.py`, **completely
unmodified**, exactly as its README says (Cell 8) -- so the AUC it prints
there is STG-NF's own real, untouched number: our first, independent sanity
check against the published checkpoint-filename numbers, computed by code
we did not touch at all.

In [ ]:
%%writefile /content/STG-NF/sentrix_score_dump.py
'''
sentrix_score_dump.py -- written for the CALM-VAD paper's STG-NF baseline
reproduction, NOT part of STG-NF's own repo. Loads a checkpoint that was
just produced by STG-NF's own UNMODIFIED train_eval.py (same --dataset /
--seg_len flags) and dumps the per-clip, per-frame raw score array that
train_eval.py already computes internally (via get_dataset_scores +
smooth_scores) to print its own AUC -- just also saved to disk here instead
of only printed and discarded, so calm.external_scores can score it.

Everything scoring-related below (get_dataset_scores, smooth_scores, the
model/dataset/trainer construction) is STG-NF's own code, imported
unmodified. The only new logic is: recovering each score array's clip name
(from the same ground-truth filename listing get_dataset_scores itself
iterates, in the same sorted order) and writing the JSON.
'''
import json
import os

import numpy as np

from args import init_parser, init_sub_args
from dataset import get_dataset_and_loader
from models.STG_NF.model_pose import STG_NF
from models.training import Trainer
from utils.data_utils import trans_list
from utils.optim_init import init_optimizer, init_scheduler
from utils.train_utils import init_model_params, calc_num_of_params
from utils.scoring_utils import get_dataset_scores, smooth_scores


def main():
    parser = init_parser()
    parser.add_argument('--out', required=True, help='output json path')
    args = parser.parse_args()
    args, model_args = init_sub_args(args)
    args.ckpt_dir = None

    dataset, loader = get_dataset_and_loader(args, trans_list=trans_list, only_test=True)
    model_args = init_model_params(args, dataset)
    model = STG_NF(**model_args)
    calc_num_of_params(model)
    trainer = Trainer(args, model, loader['train'], loader['test'],
                       optimizer_f=init_optimizer(args.model_optimizer, lr=args.model_lr),
                       scheduler_f=init_scheduler(args.model_sched, lr=args.model_lr, epochs=args.epochs))
    trainer.load_checkpoint(args.checkpoint)

    normality_scores = trainer.test()
    gt_arr, scores_arr = get_dataset_scores(normality_scores, dataset['test'].metadata, args=args)
    scores_arr = smooth_scores(scores_arr)

    if args.dataset == 'UBnormal':
        pose_segs_root = args.pose_path['test']
        clip_list = sorted(
            fn.replace('alphapose_tracked_person.json', 'tracks.txt')
            for fn in os.listdir(pose_segs_root) if fn.endswith('.json'))

        def clip_name(fn):
            return fn.replace('_tracks.txt', '')
    else:
        per_frame_scores_root = 'data/ShanghaiTech/gt/test_frame_mask/'
        clip_list = sorted(fn for fn in os.listdir(per_frame_scores_root) if fn.endswith('.npy'))

        def clip_name(fn):
            return fn.replace('.npy', '')

    assert len(clip_list) == len(scores_arr), (
        'clip_list (%d) and scores_arr (%d) length mismatch -- get_dataset_scores must '
        'have skipped some clips (only expected for --dataset ShanghaiTech-HR, which we '
        'do not use); do NOT trust the name<->score pairing below if this assertion fires'
        % (len(clip_list), len(scores_arr)))

    out = {}
    for fn, arr in zip(clip_list, scores_arr):
        name = clip_name(fn)
        # STG-NF's own scale: higher = more NORMAL (a log-likelihood). Flip so
        # higher = more ANOMALOUS, matching calm's convention everywhere else.
        out[name] = (-1.0 * np.asarray(arr, dtype=float)).tolist()

    with open(args.out, 'w', encoding='utf-8') as f:
        json.dump(out, f)
    print('wrote %d clips -> %s' % (len(out), args.out))


if __name__ == '__main__':
    main()


## Cell 8 -- train (unmodified STG-NF) + dump scores, per dataset

Resumable at whole-dataset granularity: if a dataset's checkpoint is
already on Drive, training is skipped entirely and we jump straight to
using it. **This is coarser than this project's other two Colab
notebooks** -- STG-NF's own `Trainer.train()` does not support resuming
mid-training (verified: it always starts its epoch loop at 0, even when a
checkpoint was loaded first) -- so a disconnect *during* one dataset's
training run does lose that run's progress and needs a full restart for
that dataset (default 8 epochs; see Cell 0's runtime note). The moment
training for a dataset finishes, its checkpoint is copied to Drive
immediately, before moving to the next dataset -- so once you see "copied
checkpoint" for ShanghaiTech, re-running this notebook will never retrain
it even if UBnormal (or anything after) disconnects.

In [ ]:
import glob as _glob
import shutil
import time as _time

assert GPU_OK, 'no GPU -- see Cell 1 diagnostic. STG-NF hardcodes --device cuda:0.'

BAR = '=' * 70
for key, spec in STGNF_DATASETS.items():
    ds = spec['stgnf_name']
    ckpt_dst = f'{CKPT_DIR}/{key}_checkpoint.pth.tar'
    scores_dst = f'{SCORES_DIR}/{key}_raw_scores.json'
    print(f'\n{BAR}\n  {key}  ({ds})\n{BAR}')

    seg_len_flag = f'--seg_len {spec["seg_len"]}' if spec['seg_len'] else ''

    if os.path.isfile(ckpt_dst):
        print(f'checkpoint already on Drive ({ckpt_dst}) -- skipping training')
    else:
        t0 = _time.time()
        train_cmd = f'python train_eval.py --dataset {ds} {seg_len_flag}'.strip()
        sh_stream(train_cmd, cwd=STGNF_DIR)
        elapsed_min = (_time.time() - t0) / 60.0
        print(f'\n{ds} training finished in {elapsed_min:.1f} min '
              f'(Cell 0 estimated "well under an hour" -- now you know the real number)')

        exp_dirs = sorted(_glob.glob(f'{STGNF_DIR}/data/exp_dir/{ds}/*/'), key=os.path.getmtime)
        assert exp_dirs, f'no experiment dir found under data/exp_dir/{ds}/ after training'
        ckpts = sorted(_glob.glob(f'{exp_dirs[-1]}/*_checkpoint.pth.tar'), key=os.path.getmtime)
        assert ckpts, f'no checkpoint file found in {exp_dirs[-1]} after training'
        shutil.copy(ckpts[-1], ckpt_dst)
        print(f'copied checkpoint -> {ckpt_dst}  (safe from here on, even if a later '
              'dataset disconnects)')

    if os.path.isfile(scores_dst):
        print(f'raw scores already on Drive ({scores_dst}) -- skipping score-dump')
    else:
        dump_cmd = (f'python sentrix_score_dump.py --dataset {ds} {seg_len_flag} '
                    f'--checkpoint {ckpt_dst} --out /content/{key}_raw_scores.json').strip()
        sh_stream(dump_cmd, cwd=STGNF_DIR)
        shutil.copy(f'/content/{key}_raw_scores.json', scores_dst)
        print(f'copied raw per-frame scores -> {scores_dst}')

print('\nAll datasets trained and scored (or already were).')


## Cell 9 -- score STG-NF through calm's own protocol

This is the actual point of the notebook: `calm/external_scores.py`
(added alongside this notebook, in the same branch) takes the raw per-frame
scores dumped above and puts them through the *exact same* calib/test split
rule and M3-calibration-method selection `calm.harness.run()` uses for
CALM-VAD and B0/B1/B2/B4, then reports frame AUC, event-F1 @ tIoU
{0.2..0.5}, FAPH @ recall {0.7, 0.8, 0.9}, and ECE/adaptive-ECE/Brier (raw
vs. calibrated) -- the same axes `paper/main.tex`'s results tables use, not
just frame AUC.

Ground truth (`data/pose/{shanghaitech,ubnormal}.json`) is this project's
own, already-verified file -- not the STG-NF bundle's copy -- for the
reason given in Cell 4: this project already found and fixed one
duplicate-clip bug and one ground-truth-polarity bug in its own history,
and there is no reason to re-import either bug class through a second,
independent conversion of the same underlying data.

`calm.external_scores` deliberately does not evaluate M1/M4/B0/B1/B2 for
STG-NF -- those are CALM-VAD's own front end (cue extraction, reliability
discounting, risk control) and have no equivalent for an opaque external
score. What it reports is exactly the subset that is a fair, like-for-like
comparison.

In [ ]:
os.chdir('/content/sentrix')   # back to the sentrix repo -- calm.* imports, results/ live here

for key, spec in STGNF_DATASETS.items():
    scores_path = f'/content/{key}_raw_scores.json'
    if not os.path.isfile(scores_path):
        # cell 8 already copied it to Drive; recover the local copy from there if this
        # cell is run in a fresh session after a disconnect
        import shutil
        shutil.copy(f'{SCORES_DIR}/{key}_raw_scores.json', scores_path)

    cmd = (f'python -m calm.external_scores --clips {spec["clips_json"]} '
           f'--scores {scores_path} --tag {spec["tag"]} --label "{spec["label"]}"')
    sh_stream(cmd)

    report_path = f'results/calm_report_{spec["tag"]}.json'
    with open(report_path, encoding='utf-8') as f:
        report = json.load(f)
    reproduced_auc = report['streams'][0]['frame_auc'] * 100
    print(f'\n--- {key} sanity check ---')
    print(f'  published (checkpoint filename) frame AUC : {spec["published_auc"]:.1f}%')
    print(f'  this run\'s reproduced frame AUC            : {reproduced_auc:.1f}%')
    print('  (different calib/test split and smoothing region than the published protocol,')
    print('   so an exact match is not expected -- but a large gap (many points) would be')
    print('   a red flag worth investigating before trusting the event-F1/FAPH/ECE numbers)')


## Done -- what you have, and what's next

**Outputs, per dataset:**
- `results/calm_report_stgnf_shanghaitech.{json,txt}` /
  `results/calm_report_stgnf_ubnormal.{json,txt}` -- frame AUC, event-F1,
  FAPH, ECE, computed under `calm`'s own protocol, directly comparable to
  `results/calm_report_shanghaitech.json` / `..._ubnormal.json` (CALM-VAD +
  B0/B1/B2/B4 on the same clips).
- `/content/<key>_raw_scores.json` (also on Drive under `SCORES_DIR`, i.e.
  `/content/drive/MyDrive/stgnf_baseline/scores/<key>_raw_scores.json`) --
  STG-NF's raw per-frame score, sign-flipped, before calibration. Kept in
  case you want to recompute anything (a different calibration method, a
  different tIoU set) without retraining.
- Trained checkpoints on Drive under `CKPT_DIR`, i.e.
  `/content/drive/MyDrive/stgnf_baseline/checkpoints/<key>_checkpoint.pth.tar`.

**Before using these numbers in the paper:** re-read the honesty notes in
Cell 0 -- specifically, confirm the Cell 5 alignment check actually passed
for your run, and look at the sanity-check gap printed in Cell 9. If either
looks wrong, that is more informative than the numbers themselves.

**Extending to Avenue / CHAD / NWPU-Campus (not built here, scope note):**
- **Avenue**: no native STG-NF support and no official STG-NF-format pose
  release for it. This project already has YOLO11n-pose + ByteTrack poses
  in `data/pose/avenue.json` (17-keypoint COCO, same skeleton convention),
  so a converter from that generic schema into STG-NF's native person-major
  JSON is plausible -- but unlike ShanghaiTech/UBnormal here, there is no
  STG-NF-native *training* split for Avenue to fall back on, so training
  would need Avenue's own normal-only training videos pose-extracted the
  same way. Moderate extra work, not a small tweak.
- **CHAD**: ships its own native 17-keypoint COCO pose/bbox pickles
  (`data/pose/chad.json`) but in a different container format than STG-NF's
  person-major JSON -- same category of converter work as Avenue, plus CHAD
  was never one of STG-NF's own evaluated benchmarks (no published number
  to sanity-check against at all).
- **NWPU-Campus**: same story as Avenue/CHAD (own YOLO11n-pose extraction
  in `data/pose/nwpucampus.json`, no STG-NF-native release, no published
  STG-NF number on this benchmark to check against), plus it is already
  this project's most expensive dataset to obtain (see
  `colab/calm_vad_nwpu_colab.ipynb`) -- lowest priority of the three.

All three are real, doable follow-ups (same converter shape each time: our
generic JSON -> STG-NF's person-major JSON, reconstructing the `"scores"`
confidence field honestly since none of these three releases has an
original STG-NF-format file to fall back on the way ShanghaiTech/UBnormal
do here) -- just out of scope for this notebook, which was asked to get
ShanghaiTech and UBnormal right first since those are STG-NF's own
natively-supported, highest-confidence benchmarks.

In [ ]:
from google.colab import files

for key, spec in STGNF_DATASETS.items():
    files.download(f'results/calm_report_{spec["tag"]}.json')
    files.download(f'results/calm_report_{spec["tag"]}.txt')
    files.download(f'/content/{key}_raw_scores.json')
